# CQAS Attack Simulation Results

Analysis of EPT (External Proficiency Testing) attack simulation outcomes.

In [ ]:
import sys
sys.path.insert(0, '..')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from src.detection.rule_engine import RuleEngine
from src.detection.correlation import CorrelationEngine
from src.validation.ept_engine import EPTEngine

## 1. Run All Attack Simulations

In [ ]:
rule_engine = RuleEngine('../configs/siem_rules.yaml')
correlation_engine = CorrelationEngine()
ept = EPTEngine(rule_engine, correlation_engine, '../configs/attack_simulation.yaml', dry_run=True)
ept.run_all()
summary = ept.summary()
print(f'Scenarios run: {summary["total_scenarios"]}')
print(f'Passed: {summary["passed"]} | Failed: {summary["failed"]}')
print(f'Overall coverage: {summary["coverage_percent"]:.1f}%')

## 2. Scenario Detection Rate Chart

In [ ]:
scenarios = [r['scenario_name'] for r in summary['results']]
detection_rates = [r['detection_rate'] for r in summary['results']]
colors = ['green' if r == 100 else 'orange' if r >= 50 else 'red' for r in detection_rates]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(scenarios, detection_rates, color=colors, alpha=0.85, edgecolor='black', linewidth=0.5)
ax.axhline(100, color='green', linestyle='--', alpha=0.5, label='Full coverage (100%)')
ax.axhline(80, color='orange', linestyle='--', alpha=0.5, label='Acceptable (80%)')
for bar, rate in zip(bars, detection_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{rate:.0f}%', ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0, 115)
ax.set_ylabel('Detection Rate (%)')
ax.set_title('EPT Scenario Detection Rates')
ax.tick_params(axis='x', rotation=30)
green_p = mpatches.Patch(color='green', label='100% detected')
orange_p = mpatches.Patch(color='orange', label='Partial detection')
red_p = mpatches.Patch(color='red', label='Not detected')
ax.legend(handles=[green_p, orange_p, red_p])
plt.tight_layout()
plt.show()

## 3. MITRE ATT&CK Coverage Heatmap

In [ ]:
# Simulated MITRE ATT&CK coverage matrix
tactics = ['Recon', 'Resource\nDev', 'Initial\nAccess', 'Execution', 'Persistence',
           'Priv Esc', 'Defense\nEvasion', 'Cred\nAccess', 'Discovery', 'Lateral\nMove',
           'Collection', 'C2', 'Exfil', 'Impact']
# Coverage: 0=none, 1=partial, 2=full
coverage = [1, 0, 0, 0, 0, 1, 0, 2, 2, 2, 0, 2, 2, 0]
cmap = plt.cm.get_cmap('RdYlGn', 3)
fig, ax = plt.subplots(figsize=(14, 3))
im = ax.imshow([coverage], cmap=cmap, vmin=0, vmax=2, aspect='auto')
ax.set_xticks(range(len(tactics)))
ax.set_xticklabels(tactics, fontsize=9)
ax.set_yticks([])
ax.set_title('MITRE ATT&CK Tactic Coverage (0=None, 1=Partial, 2=Full)')
cbar = plt.colorbar(im, ax=ax, orientation='vertical', shrink=0.8)
cbar.set_ticks([0, 1, 2])
cbar.set_ticklabels(['None', 'Partial', 'Full'])
plt.tight_layout()
plt.show()